# 1. Paquetes y carga del dataset

In [26]:
# Cargar librerías
from IPython.display import display
import pyreadstat
import re
import unicodedata
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import prince
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

# Cargar el archivo .sav
df, meta = pyreadstat.read_sav("/Users/tomas/github/eaui_subtel/data/sav/2026.sav")

# Pasar nombres de columnas a minúscula
df.columns = df.columns.str.lower()

# Eliminar espacios en blanco al inicio y al final de los nombres de columnas
df.columns = df.columns.str.strip()

# verificar los nombres de las columnas
print(df.columns)

# Verificar la cantidad de filas y columnas
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")

Index(['registro', 'fechafin', 'cod_region', 'comuna_def', 'zona', 'a9', 'a10',
       'a11', 'a12_11', 'a12_2',
       ...
       'q40_3', 'q40_4', 'q40_5', 'q42', 'q42_1', 'a12_1', 'pond_hogar',
       'fe_hogar', 'pond_personas', 'fe_personas'],
      dtype='str', length=587)
Filas: 5,000 | Columnas: 587


# 3. GSE

In [27]:
def _educ_g(e):
    if pd.isna(e): return None
    e = int(e)
    if e <= 3:  return 'basica'
    if e <= 7:  return 'media'
    if e <= 9:  return 'tecnica'
    return 'universitaria'

_M = {
    (1,'basica'):'E',  (1,'media'):'E',  (1,'tecnica'):'D',  (1,'universitaria'):'D',
    (2,'basica'):'E',  (2,'media'):'D',  (2,'tecnica'):'D',  (2,'universitaria'):'C3',
    (3,'basica'):'D',  (3,'media'):'C3', (3,'tecnica'):'C3', (3,'universitaria'):'C2',
    (4,'basica'):'C3', (4,'media'):'C2', (4,'tecnica'):'C2', (4,'universitaria'):'C1',
    (5,'basica'):'C2', (5,'media'):'C1', (5,'tecnica'):'C1', (5,'universitaria'):'AB',
    (6,'basica'):'C1', (6,'media'):'AB', (6,'tecnica'):'AB', (6,'universitaria'):'AB',
}
_ORDEN_GSE = ['AB','C1','C2','C3','D','E']  # Invertido

_eg = df['a10'].apply(_educ_g)
df['gse'] = pd.Categorical(
    df['a11'].combine(_eg, lambda o, e: np.nan if pd.isna(o) or e is None else _M.get((int(o), e), np.nan)),
    categories=_ORDEN_GSE, ordered=True  # Aquí usa el orden invertido automáticamente
)
print('GSE:', df['gse'].value_counts().reindex(_ORDEN_GSE).to_dict())

display(df['gse'].value_counts().reindex(_ORDEN_GSE))

GSE: {'AB': 342, 'C1': 533, 'C2': 988, 'C3': 1316, 'D': 833, 'E': 988}


gse
AB     342
C1     533
C2     988
C3    1316
D      833
E      988
Name: count, dtype: int64

# 4. Etiquetado de variables

In [ ]:
etiquetas_limpias = {
    col.lower().strip(): limpiar_etiqueta(label)
    for col, label in zip(meta.column_names, meta.column_labels) if label
}
print(f"Etiquetas procesadas: {len(etiquetas_limpias)}")

# 5. Tratamiento de NS/NR

In [28]:
cols_nsnr = [
    'p11','q7_4',
    'p17_1','p17_2','p17_3','p17_4','p17_5',
    'p19_1','p19_2','p19_3','p19_4',
    'q40_1','q40_2','q40_3','q40_4','q40_5',
    'q42','q42_1'
]
for col in cols_nsnr:
    if col in df.columns: df[col] = df[col].replace(9999999, np.nan)
print('NS/NR reemplazados por NaN.')

NS/NR reemplazados por NaN.


# 6. Renombrado de variables

In [29]:
nombres_cortos = {
    'registro':'id', 'fechafin':'fecha_fin', 'cod_region':'region', 'comuna_def':'comuna', 'zona':'zona',
    'a9':'parentesco_jh', 'a10':'educ_jh', 'a11':'ocupacion_jh', 'a12_1':'ingreso_hogar',
    'q1':'parentesco', 'q1_1':'edad', 'q1_2':'sexo', 'q1_3':'educ', 'q1_4':'ocupacion_encuestado', 'q2':'actividad',
    'p1':'acceso_internet_hogar', 'p2':'n_smartphones_hogar', 'p2_1':'n_computadores_hogar',
    'p10':'tipo_acceso_fijo', 'p11':'pago_mensual_internet', 'p11_3':'velocidad_contratada',
    'p11_4':'calidad_acceso', 'p11_5':'cuota_mensual_gb', 'p12_2':'tipo_plan', 'p12_1':'plan_movil_tipo',
    'p14':'razon_no_acceso_principal', 'p15':'disposicion_contratar_fijo',
    'q5':'uso_computador', 'q7':'uso_smartphone', 'q7_1':'smartphone_propio',
    'q7_3':'plan_movil_tipo_ind', 'q7_4':'pago_mensual_movil',
    'q9':'ultimo_uso_internet', 'q10':'frecuencia_internet', 'q11':'tiempo_diario_internet',
    'q13':'tipo_acceso_mas_usado', 'q14':'uso_internet_hogar', 'q15':'frecuencia_internet_hogar',
    'q16':'tiempo_diario_hogar', 'q17':'uso_internet_fuera_hogar', 'q18':'frecuencia_fuera_hogar',
    'q19':'tiempo_diario_fuera_hogar',
    'q23':'internet_facilita_trabajo', 'q25':'internet_mejora_vida', 'q27':'ultima_compra_online',
    'q31':'percepcion_proteccion', 'q30_1':'reg_control_legal', 'q30_2':'reg_control_familia', 'q30_3':'reg_autocontrol',
    'fe_hogar':'fe_hogar', 'fe_personas':'fe_personas', 'pond_hogar':'pond_hogar', 'pond_personas':'pond_personas',
}

df = df.rename(columns={k: v for k, v in nombres_cortos.items() if k in df.columns})

# Recodificación de educ_jh y ocupacion_jh (aquí, con valores numéricos aún intactos)
_mapa_educ = {
    1:'Sin educación formal', 2:'Básica incompleta', 3:'Básica completa',
    4:'Media CH incompleta', 5:'Media TP incompleta', 6:'Media CH completa', 7:'Media TP completa',
    8:'Superior técnica incompleta', 9:'Superior técnica completa',
    10:'Superior universitaria incompleta', 11:'Superior universitaria completa'
}
_mapa_ocup = {
    1:'Trabajos ocasionales e informales', 2:'Oficio menor - obrero no calificado',
    3:'Obrero calificado - microempresario', 4:'Empleado medio - técnico - prof. independiente',
    5:'Ejecutivo medio - prof. universitario', 6:'Alto ejecutivo - empresario - directivo'
}
df['educ_jh']              = df['educ_jh'].map(_mapa_educ)
df['ocupacion_jh']         = df['ocupacion_jh'].map(_mapa_ocup)
df['ocupacion_encuestado'] = df['ocupacion_encuestado'].map({**_mapa_ocup, 7:'Sin trabajo remunerado'})

print(f"Renombradas: {len(nombres_cortos)} | Columnas totales: {df.shape[1]}")

Renombradas: 53 | Columnas totales: 588


# 7. Recodificación de variables

In [30]:
df = df.copy()

# Identificación
df['region'] = df['region'].map({
    1:'Tarapacá', 2:'Antofagasta', 3:'Atacama', 4:'Coquimbo', 5:'Valparaíso',
    6:"O'Higgins", 7:'Maule', 8:'Biobío', 9:'Araucanía', 10:'Los Lagos',
    11:'Aysén', 12:'Magallanes', 13:'Metropolitana', 14:'Los Ríos', 15:'Arica y Parinacota', 16:'Ñuble'
})
df['zona'] = df['zona'].map({1:'Urbana', 2:'Rural'})

# Sociodemográficas del entrevistado
df['sexo'] = df['sexo'].map({1:'Hombre', 2:'Mujer'})
df['educ'] = df['educ'].map(_mapa_educ)
df['educ_grupo'] = df['educ'].map({
    'Sin educación formal':'Básica o menos', 'Básica incompleta':'Básica o menos',
    'Básica completa':'Básica o menos', 'Media CH incompleta':'Media',
    'Media TP incompleta':'Media', 'Media CH completa':'Media', 'Media TP completa':'Media',
    'Superior técnica incompleta':'Superior', 'Superior técnica completa':'Superior',
    'Superior universitaria incompleta':'Superior', 'Superior universitaria completa':'Superior',
})
df['tramo_edad'] = pd.cut(df['edad'], bins=[0,17,29,44,59,200],
                          labels=['Menor de 18','18-29','30-44','45-59','60 y más'], right=True)
df['actividad'] = df['actividad'].map({
    1:'Trabajador independiente', 2:'Empleador/patrón', 3:'Empleado dependiente',
    4:'Familiar no remunerado', 5:'FFAA y de orden', 6:'Cesante',
    7:'Jubilado/pensionado', 8:'Estudiante', 9:'Labores del hogar'
})

# Acceso a internet
df['acceso_internet_hogar'] = df['acceso_internet_hogar'].map({1:'Sí', 2:'No'})
df['tipo_acceso_fijo'] = df['tipo_acceso_fijo'].map({
    1:'ADSL', 2:'Cable/Módem', 3:'Fibra óptica', 4:'Inalámbrica',
    5:'Satelital', 31:'WiFi', 32:'Antena', 33:'Banda ancha', 34:'Acceso telefónico', 88:'No sabe'
})
df['velocidad_contratada'] = df['velocidad_contratada'].map({
    1:'Hasta 10 Mbps', 2:'Más de 10 a 100 Mbps', 3:'Más de 100 a 500 Mbps',
    4:'Más de 500 Mbps a 1 Gbps', 5:'Más de 1 Gbps', 99:'NS/NR'
})
df['tipo_plan'] = df['tipo_plan'].map({
    1:'Banda ancha desnuda', 2:'BA + TV Cable', 3:'BA + Telefonía fija',
    4:'Triple pack (BA+TV+Tel)', 5:'Otros planes'
})
df['tipo_acceso_mas_usado'] = df['tipo_acceso_mas_usado'].map({
    1.0:'Banda Ancha Fija / WiFi', 2.0:'Banda Ancha Móvil',
    3.0:'Internet Móvil (Smartphone/Tablet)', 4.0:'Conexión Satelital'
})

# Uso individual
df['uso_computador']  = df['uso_computador'].map({1:'Sí', 2:'No'})
df['uso_smartphone']  = df['uso_smartphone'].map({1:'Sí', 2:'No'})
df['ultimo_uso_internet'] = df['ultimo_uso_internet'].map({
    1:'Hoy', 2:'Entre 2 y 3 días', 3:'Entre 3 y 7 días', 4:'Entre 1 y 4 semanas',
    5:'Más de 4 semanas', 6:'Más de 12 meses', 7:'Nunca'
})
df['frecuencia_internet'] = df['frecuencia_internet'].map({
    1:'Todos los días', 2:'Varias veces por semana',
    3:'Al menos una vez al mes', 4:'Menos de una vez al mes'
})
df['tiempo_diario_internet'] = df['tiempo_diario_internet'].map({
    1:'Menos de 1 hora', 2:'Entre 1 y 2 horas', 3:'Entre 2 y 4 horas', 4:'Más de 4 horas'
})

# Percepciones
df['percepcion_proteccion']     = df['percepcion_proteccion'].map({
    1:'Muy protegido', 2:'Protegido', 3:'Desprotegido', 4:'Muy desprotegido', 99:'NS/NR'
})
df['internet_mejora_vida']      = df['internet_mejora_vida'].map({1:'Sí', 2:'No'})
df['internet_facilita_trabajo'] = df['internet_facilita_trabajo'].map({1:'Sí', 2:'No'})

print('Recodificaciones completadas.')
print(f"sexo: {df['sexo'].value_counts().to_dict()}")
print(f"acceso: {df['acceso_internet_hogar'].value_counts().to_dict()}")

Recodificaciones completadas.
sexo: {'Mujer': 2815, 'Hombre': 2185}
acceso: {'Sí': 4841, 'No': 159}


# 8. Ingreso hogar

In [31]:
_rangos = {
    11:(0,129000),12:(130000,226000),13:(227000,393000),14:(394000,686000),15:(687000,1100000),16:(1200000,2000000),17:(2100000,None),
    21:(0,210000),22:(211000,366000),23:(367000,639000),24:(640000,1100000),25:(1200000,1900000),26:(2000000,3300000),27:(3400000,None),
    31:(0,279000),32:(280000,487000),33:(488000,849000),34:(850000,1400000),35:(1500000,2500000),36:(2600000,4500000),37:(4600000,None),
    41:(0,341000),42:(342000,595000),43:(596000,1000000),44:(1100000,1800000),45:(1900000,3100000),46:(3200000,5500000),47:(5600000,None),
    51:(0,399000),52:(400000,696000),53:(697000,1200000),54:(1300000,2100000),55:(2200000,3600000),56:(3700000,6400000),57:(6500000,None),
    61:(0,453000),62:(454000,791000),63:(792000,1300000),64:(1400000,2400000),65:(2500000,4100000),66:(4200000,7300000),67:(7400000,None),
    71:(0,505000),72:(506000,881000),73:(882000,1500000),74:(1600000,2600000),75:(2700000,4600000),76:(4700000,8100000),77:(8200000,None),
    81:(0,555000),82:(556000,967000),83:(968000,1600000),84:(1700000,2900000),85:(3000000,5100000),86:(5200000,8900000),87:(9000000,None),
    91:(0,602000),92:(603000,1000000),93:(1100000,1800000),94:(1900000,3100000),95:(3200000,5500000),96:(5600000,9700000),97:(9800000,None),
    101:(0,648000),102:(649000,1100000),103:(1200000,1900000),104:(2000000,3400000),105:(3500000,5900000),106:(6000000,10400000),107:(10500000,None),
}
_mapa_pm = {float(k): (v[0]*1.5 if v[1] is None else (v[0]+v[1])/2) for k, v in _rangos.items()}

df['ingreso_pm'] = df['ingreso_hogar'].map(_mapa_pm)
df['ingreso_tramo'] = pd.cut(
    df['ingreso_pm'],
    bins=[0, 384000, 540000, 798000, 1100000, 1700000, float('inf')],
    labels=['Hasta $384 mil','$384 mil a $540 mil','$540 mil a $798 mil',
            '$798 mil a $1,1 millón','$1,1 millón a $1,7 millones','Más de $1,7 millones'],
    right=True
)
df['ingreso_grupo'] = df['ingreso_tramo'].map({
    'Hasta $384 mil':'Bajo', '$384 mil a $540 mil':'Bajo',
    '$540 mil a $798 mil':'Medio', '$798 mil a $1,1 millón':'Medio',
    '$1,1 millón a $1,7 millones':'Alto', 'Más de $1,7 millones':'Alto',
})

# Validación: promedio de ingreso debe subir de E a AB
(
    df.groupby('gse', observed=True)['ingreso_pm']
    .agg(N='count', Promedio='mean').reindex(_ORDEN_GSE).round(0).astype({'N':int,'Promedio':int})
)

,N,Promedio
gse,,
AB,286,2097505
C1,444,1389884
C2,826,986176
C3,1112,799533
D,704,650022
E,846,539833


# 9. Ordenar categorías

In [219]:
ORDEN_CATEGORIAS = {
    'sexo':         ['Hombre','Mujer'],
    'zona':         ['Urbana','Rural'],
    'region': ['Arica y Parinacota','Tarapacá','Antofagasta','Atacama','Coquimbo','Valparaíso','Metropolitana',
               "O'Higgins",'Maule','Ñuble','Biobío','Araucanía','Los Ríos','Los Lagos','Aysén','Magallanes'],
    'educ':         ['Sin educación formal','Básica incompleta','Básica completa',
                     'Media CH incompleta','Media TP incompleta','Media CH completa','Media TP completa',
                     'Superior técnica incompleta','Superior técnica completa',
                     'Superior universitaria incompleta','Superior universitaria completa'],
    'educ_grupo':   ['Básica o menos','Media','Superior'],
    'tramo_edad':   ['Menor de 18','18-29','30-44','45-59','60 y más'],
    'actividad':    ['Trabajador independiente','Empleador/patrón','Empleado dependiente',
                     'Familiar no remunerado','FFAA y de orden','Cesante',
                     'Jubilado/pensionado','Estudiante','Labores del hogar'],
    'ocupacion_jh': ['Trabajos ocasionales e informales','Oficio menor - obrero no calificado',
                     'Obrero calificado - microempresario','Empleado medio - técnico - prof. independiente',
                     'Ejecutivo medio - prof. universitario','Alto ejecutivo - empresario - directivo'],
    'ocupacion_encuestado': ['Trabajos ocasionales e informales','Oficio menor - obrero no calificado',
                     'Obrero calificado - microempresario','Empleado medio - técnico - prof. independiente',
                     'Ejecutivo medio - prof. universitario','Alto ejecutivo - empresario - directivo',
                     'Sin trabajo remunerado'],
    'gse':              ['AB', 'C1', 'C2', 'C3', 'D', 'E'],
    'ingreso_tramo':    ['Hasta $384 mil','$384 mil a $540 mil','$540 mil a $798 mil',
                         '$798 mil a $1,1 millón','$1,1 millón a $1,7 millones','Más de $1,7 millones'],
    'ingreso_grupo':    ['Bajo','Medio','Alto'],
    'acceso_internet_hogar':    ['Sí','No'],
    'uso_computador':           ['Sí','No'],
    'uso_smartphone':           ['Sí','No'],
    'internet_mejora_vida':     ['Sí','No'],
    'internet_facilita_trabajo':['Sí','No'],
    'tipo_acceso_fijo':         ['Fibra óptica','Cable/Módem','ADSL','Inalámbrica','Satelital','WiFi','Antena','Banda ancha','Acceso telefónico','No sabe'],
    'tipo_acceso_mas_usado':    ['Banda Ancha Fija / WiFi','Banda Ancha Móvil','Internet Móvil (Smartphone/Tablet)','Conexión Satelital'],
    'tipo_plan':                ['Banda ancha desnuda','BA + TV Cable','BA + Telefonía fija','Triple pack (BA+TV+Tel)','Otros planes'],
    'ultimo_uso_internet':      ['Hoy','Entre 2 y 3 días','Entre 3 y 7 días',
                                  'Entre 1 y 4 semanas','Más de 4 semanas','Más de 12 meses','Nunca'],
    'frecuencia_internet':      ['Todos los días','Varias veces por semana',
                                  'Al menos una vez al mes','Menos de una vez al mes'],
    'tiempo_diario_internet':   ['Menos de 1 hora','Entre 1 y 2 horas','Entre 2 y 4 horas','Más de 4 horas'],
    'percepcion_proteccion':    ['Muy protegido','Protegido','Desprotegido','Muy desprotegido','NS/NR'],
    'velocidad_contratada':     ['Hasta 10 Mbps','Más de 10 a 100 Mbps','Más de 100 a 500 Mbps',
                                  'Más de 500 Mbps a 1 Gbps','Más de 1 Gbps','NS/NR'],
}

# 10. Funciones para análisis descriptivo univariado

In [297]:
# Tabla de frecuencias univariada
def unifreq(dataframe, variable, factor_expansion, orden=ORDEN_CATEGORIAS):
    resultado = (dataframe.groupby(variable)[factor_expansion].sum() / 
                 dataframe[factor_expansion].sum() * 100).to_frame('porcentaje')
    resultado['porcentaje'] = resultado['porcentaje'].round(1)
    
    if orden is not None and variable in orden:
        orden_lista = orden[variable]
        resultado = resultado.reindex(orden_lista)
    
    return resultado

# Gráfico de barras univariado
def unigbar(datos, titulo_x, titulo_y='Porcentaje (%)'):
    plt.figure(figsize=(10, 6))
    ax = sns.barplot(
        data = datos,
        y = 'porcentaje',
        x = datos.index,
        color = 'darkseagreen'
    )
    
    plt.ylim(0, 100)
    plt.xlabel(titulo_x)
    plt.ylabel(titulo_y)
    plt.title(f'Frecuencia de {titulo_x}', fontsize=12, weight='bold')
    plt.xticks(rotation=45, ha='right')
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    plt.grid(False)
    
    ax.bar_label(ax.containers[0], fmt='%.1f%%')
    
    plt.show()

# Tabla de frecuencias bivariada
def bifreq(dataframe, variable1, variable2, factor_expansion):
    tabla = pd.crosstab(dataframe[variable1], dataframe[variable2], values=dataframe[factor_expansion], aggfunc='sum', normalize='index') * 100
    tabla = tabla.round(1)

    if variable1 in ORDEN_CATEGORIAS:
        tabla = tabla.reindex(ORDEN_CATEGORIAS[variable1])

    return tabla

#Gráfico de barras bivariado o apilado
def bigbar(datos, titulo_x, titulo_y='Porcentaje (%)', tipo='bivariado', cmap='viridis'):
    plt.figure(figsize=(10, 6))
    
    if tipo == 'bivariado':
        ax = datos.plot(kind='bar', ax=plt.gca(), width=0.8, colormap=cmap)
    elif tipo == 'apilado':
        ax = datos.plot(kind='bar', stacked=True, ax=plt.gca(), width=0.8, colormap=cmap)
    
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%')
    
    plt.ylim(0, 100 if tipo == 'apilado' else None)
    plt.xlabel(titulo_x)
    plt.ylabel(titulo_y)
    plt.title(f'Frecuencia de {titulo_x}', fontsize=12, weight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    plt.grid(False)
    
    plt.show()

# 11. Conjuntos de respuesta múltiple

In [315]:
_c = df.columns

GRUPOS_RM = {
    # Hogar
    'a12':  ('Pueblos indígenas o tribales (hogar)',           [c for c in _c if c.startswith('a12_') and not c.startswith('a12_1')]),
    'a13':  ('Condiciones permanentes de salud en el hogar',   [c for c in _c if c.startswith('a13_')]),
    'a14':  ('Situaciones laborales en el hogar',              [c for c in _c if c.startswith('a14_') and not c.endswith('_OTRA')]),
    # Acceso y conectividad
    'p3':   ('Dispositivos usados para acceder a internet',    [c for c in _c if c.startswith('p3_') and not c.endswith('_OTRA')]),
    'p4':   ('Formas de acceso pagado a internet en el hogar', [c for c in _c if c.startswith('p4_')]),
    'p6':   ('Razones para tener internet fijo',               [c for c in _c if c.startswith('p6_') and not c.startswith('p6_1_') and not c.endswith('_OTRA')]),
    'p6_1': ('Razones para tener internet móvil',              [c for c in _c if c.startswith('p6_1_')]),
    'p7':   ('Medidas de protección internet para menores',    [c for c in _c if c.startswith('p7_')]),
    'p9':   ('Dispositivos de uso personal de menores',        [c for c in _c if c.startswith('p9_')]),
    'p12':  ('Conexión móvil 4G/5G',                           ['p12_11','p12_21','p12_31','p12_41']),
    'p13':  ('Razones de no acceso a internet fijo',           [c for c in _c if c.startswith('p13_') and not c.endswith('_OTRA')]),
    'p16':  ('Equipos que le interesaría tener (sin internet)',[c for c in _c if c.startswith('p16_')]),
    # Uso individual
    'q6':   ('¿Cómo aprendió a usar el computador?',           [c for c in _c if c.startswith('q6_') and c not in ['q6_1','q6_OTRA']]),
    'q7_2': ('Smartphone 4G/5G',                               ['q7_2_1','q7_2_2','q7_2_3','q7_2_4']),
    'q8':   ('Habilidades digitales',                          [c for c in _c if c.startswith('q8_')]),
    'q11_1':('Lugares donde usó internet ayer',                [c for c in _c if c.startswith('q11_1_')]),
    'q12':  ('Tipos de acceso en últimos 3 meses',             [c for c in _c if c.startswith('q12_')]),
    'q20':  ('Lugares donde usó internet fuera del hogar',     [c for c in _c if c.startswith('q20_')]),
    'q21':  ('Actividades realizadas en internet',             [c for c in _c if c.startswith('q21_') and c not in ['q21_1','q21_10','q21_19','q21_26','q21_33','q21_38','q21_44']]),
    'q28':  ('Bienes y servicios comprados en internet',       [c for c in _c if c.startswith('q28_') and not c.endswith('_OTRA')]),
    'q32':  ('Actividades de seguridad y privacidad',          [c for c in _c if c.startswith('q32_') and not c.endswith('_OTRA')]),
    'q33':  ('Problemas de seguridad sufridos',                [c for c in _c if c.startswith('q33_') and not c.endswith('_OTRA')]),
    'q34':  ('Razones de no uso de internet',                  [c for c in _c if c.startswith('q34_') and not c.endswith('_OTRA')]),
    'q37':  ('Actividades de internet realizadas por terceros',[c for c in _c if c.startswith('q37_')]),
    'q39':  ('Equipos que le interesaría tener (no usuarios)', [c for c in _c if c.startswith('q39_')]),
}

def crm(grupo, factor='fe_hogar', top_n=None):
    """
    Analiza un grupo de respuesta múltiple.
    """
    if grupo not in GRUPOS_RM: 
        raise ValueError(f"Grupo '{grupo}' no existe.")
        
    desc, cols = GRUPOS_RM[grupo]
    cols = [c for c in cols if c in df.columns]
    
    if not cols:
        print(f"No hay columnas válidas para '{grupo}'")
        return None
    
    base = df.loc[df[cols].notna().any(axis=1), factor].sum()
    
    filas = [
        {'variable': c,
         'etiqueta': etiquetas_limpias.get(c, c),
         #'n_ponderado': int(df.loc[df[c]==1, factor].sum()),
         'porcentaje': round(df.loc[df[c]==1, factor].sum() / base * 100, 1)}
        for c in cols
    ]
    
    if not filas:
        print(f"Sin datos para '{grupo}'")
        return None
    
    res = pd.DataFrame(filas).sort_values('porcentaje', ascending=False).reset_index(drop=True)
    if top_n: res = res.head(top_n)
    res.index += 1
    
    return res

In [316]:
crm('q12', factor='fe_hogar')

,variable,etiqueta,porcentaje
1,q12_3,q12_3,86.3
2,q12_1,q12_1,66.8
3,q12_2,q12_2,4.9
4,q12_4,q12_4,1.2
5,q12_5,q12_5,0.4
